<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/10_SPP_GAN_Differential_Privacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ==================================================================================================
# 1. HEADER & SCOPE
# ==================================================================================================

print("\n" + "=" * 100)
print("1. HEADER & SCOPE")
print("=" * 100)

from pathlib import Path

NOTEBOOK_ID = "10"
NOTEBOOK_NAME = "SPP-GAN Differential Privacy"
FRAMEWORK_NAME = "SPP-GAN"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NOTEBOOK_SCOPE = {
    "purpose": (
        "Define and validate the differential privacy mechanism "
        "for the SPP-GAN discriminator."
    ),
    "protected_component": "SPP-GAN discriminator",
    "privacy_mechanism": "DP-SGD",
    "gradient_mechanism": "per-example discriminator gradients",
    "clipping_mechanism": "flat L2 clipping",
    "noise_mechanism": "Gaussian",
    "sampling_mechanism": "Poisson",
    "accountant": "RDP",
    "training_performed": False,
    "accounting_performed": False,
    "synthetic_generation_performed": False,
    "end_to_end_privacy_claim": False,
}

PRIVACY_BOUNDARY = {
    "discriminator_update_private": True,
    "generator_update_private_in_notebook_10": False,
    "statistical_guidance_private_in_notebook_10": False,
    "preprocessing_private_in_notebook_10": False,
    "conditioning_private_in_notebook_10": False,
    "formal_accounting_notebook": "11",
    "training_notebook": "12",
    "generation_notebook": "13",
}

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Canonical project root not found:\n{PROJECT_ROOT}"
    )

print(f"Notebook ID                    : {NOTEBOOK_ID}")
print(f"Notebook                       : {NOTEBOOK_NAME}")
print(f"Framework                      : {FRAMEWORK_NAME}")
print(f"Project root                   : {PROJECT_ROOT}")
print(f"Privacy mechanism              : DP-SGD")
print(f"Protected component            : SPP-GAN discriminator")
print(f"Gradient mechanism             : per-example gradients")
print(f"Clipping mechanism             : flat L2")
print(f"Noise mechanism                : Gaussian")
print(f"Sampling mechanism             : Poisson")
print(f"Privacy accountant             : RDP")

print()
print("Privacy boundary:")
print("  ✓ Discriminator update       : PRIVATE")
print("  ✓ Statistical guidance       : NOT PRIVATIZED IN NB10")
print("  ✓ Preprocessing              : NOT PRIVATIZED IN NB10")
print("  ✓ Generator update           : NOT PRIVATIZED IN NB10")
print("  ✓ Formal accounting          : DEFERRED TO NB11")
print("  ✓ SPP-GAN training           : DEFERRED TO NB12")
print("  ✓ Synthetic generation       : DEFERRED TO NB13")

print()
print("✓ Canonical project root verified.")
print("✓ Notebook 10 scope established.")


1. HEADER & SCOPE


FileNotFoundError: Canonical project root not found:
/content/drive/MyDrive/SPP_GAN_Research

In [6]:
# ==================================================================================================
# 2. LOAD CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("2. LOAD CONFIGURATION")
print("=" * 100)

import json
import hashlib
import math
import sys
import subprocess
import importlib.util
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# --------------------------------------------------------------------------------------------------
# Google Drive
# --------------------------------------------------------------------------------------------------

from google.colab import drive

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

if not MYDRIVE.exists():
    drive.mount("/content/drive")

if not MYDRIVE.exists():
    raise RuntimeError(
        "Google Drive is not available."
    )

print(f"✓ MyDrive        : {MYDRIVE}")

# --------------------------------------------------------------------------------------------------
# Canonical roots
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = (
    MYDRIVE /
    "SPP_GAN_Research"
)

NB08_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_08"
)

NB09_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_09"
)

NB10_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_10"
)

for name, path in {
    "Project root": PROJECT_ROOT,
    "Notebook 08": NB08_ROOT,
    "Notebook 09": NB09_ROOT,
}.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

# --------------------------------------------------------------------------------------------------
# Notebook 10 directories
# --------------------------------------------------------------------------------------------------

DIRS = {
    "root": NB10_ROOT,
    "models": NB10_ROOT / "models",
    "configuration": NB10_ROOT / "configuration",
    "metadata": NB10_ROOT / "metadata",
    "validation": NB10_ROOT / "validation",
    "manifests": NB10_ROOT / "manifests",
}

for directory in DIRS.values():
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# --------------------------------------------------------------------------------------------------
# Dataset registry
# --------------------------------------------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

if len(DATASET_IDS) != 3:
    raise RuntimeError(
        "Expected exactly three datasets."
    )

# --------------------------------------------------------------------------------------------------
# Canonical Notebook 02 training sizes
# --------------------------------------------------------------------------------------------------

TRAIN_ROWS = {
    "adult_income": 34189,
    "bank_marketing": 31647,
    "diabetes_130us": 71236,
}

# --------------------------------------------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------------------------------------------

MASTER_SEED = 2025

np.random.seed(
    MASTER_SEED
)

torch.manual_seed(
    MASTER_SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        MASTER_SEED
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"✓ Project root   : {PROJECT_ROOT}")
print(f"✓ Notebook 08    : {NB08_ROOT}")
print(f"✓ Notebook 09    : {NB09_ROOT}")
print(f"✓ Notebook 10    : {NB10_ROOT}")
print(f"✓ Master seed    : {MASTER_SEED}")
print(f"✓ Device         : {DEVICE}")

# --------------------------------------------------------------------------------------------------
# Notebook 09 configuration
# --------------------------------------------------------------------------------------------------

NB09_CONFIGURATION_PATH = (
    NB09_ROOT /
    "configuration" /
    "sppgan_statistical_guidance_configuration.json"
)

if not NB09_CONFIGURATION_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 09 configuration not found:\n"
        f"{NB09_CONFIGURATION_PATH}"
    )

with open(
    NB09_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as f:
    NB09_GUIDANCE_CONFIGURATION = json.load(f)

REQUIRED_NB09_KEYS = {
    "configuration_version",
    "objective",
    "weights",
    "distribution_guidance",
    "moment_guidance",
    "categorical_guidance",
    "dependency_guidance",
    "privacy",
}

missing_nb09_keys = (
    REQUIRED_NB09_KEYS
    -
    set(
        NB09_GUIDANCE_CONFIGURATION.keys()
    )
)

if missing_nb09_keys:
    raise RuntimeError(
        "Notebook 09 configuration missing keys:\n"
        f"{sorted(missing_nb09_keys)}"
    )

LAMBDA_STAT = float(
    NB09_GUIDANCE_CONFIGURATION[
        "objective"
    ][
        "lambda_stat"
    ]
)

if LAMBDA_STAT < 0:
    raise ValueError(
        "lambda_stat must be non-negative."
    )

print()
print(
    f"✓ Notebook 09 configuration loaded:\n"
    f"  {NB09_CONFIGURATION_PATH}"
)

print(
    f"✓ λ_stat = {LAMBDA_STAT:.6f}"
)

print("SECTION 2 STATUS: PASS")


2. LOAD CONFIGURATION


MessageError: Error: credential propagation was unsuccessful

In [7]:
# ==================================================================================================
# 3. LOAD SPP-GAN ARCHITECTURE
# ==================================================================================================

print("\n" + "=" * 100)
print("3. LOAD SPP-GAN ARCHITECTURE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Locate architecture summary
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_CANDIDATES = sorted(
    set(
        list(
            NB08_ROOT.rglob(
                "*architecture*summary*.csv"
            )
        )
        +
        list(
            NB08_ROOT.rglob(
                "*architecture_summary*.csv"
            )
        )
    ),
    key=lambda p: len(str(p))
)

if not ARCHITECTURE_CANDIDATES:
    raise FileNotFoundError(
        "Notebook 08 architecture summary not found."
    )

ARCHITECTURE_SUMMARY_PATH = (
    ARCHITECTURE_CANDIDATES[0]
)

ARCHITECTURE_SUMMARY_DF = pd.read_csv(
    ARCHITECTURE_SUMMARY_PATH
)

REQUIRED_ARCH_COLUMNS = {
    "dataset",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
}

missing_columns = (
    REQUIRED_ARCH_COLUMNS
    -
    set(
        ARCHITECTURE_SUMMARY_DF.columns
    )
)

if missing_columns:
    raise RuntimeError(
        "Architecture summary missing columns:\n"
        f"{sorted(missing_columns)}"
    )

ARCHITECTURE_SUMMARY_DF["dataset"] = (
    ARCHITECTURE_SUMMARY_DF[
        "dataset"
    ]
    .astype(str)
)

ARCHITECTURE_SUMMARY_DF = (
    ARCHITECTURE_SUMMARY_DF[
        ARCHITECTURE_SUMMARY_DF[
            "dataset"
        ].isin(
            DATASET_IDS
        )
    ]
    .copy()
)

if len(ARCHITECTURE_SUMMARY_DF) != 3:
    raise RuntimeError(
        "Architecture summary does not contain all three datasets."
    )

ARCHITECTURE_SUMMARY_DF = (
    ARCHITECTURE_SUMMARY_DF
    .set_index("dataset")
    .loc[DATASET_IDS]
    .reset_index()
)

for column in [
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
]:

    ARCHITECTURE_SUMMARY_DF[column] = pd.to_numeric(
        ARCHITECTURE_SUMMARY_DF[column],
        errors="coerce",
    )

    if ARCHITECTURE_SUMMARY_DF[column].isna().any():
        raise RuntimeError(
            f"Invalid values in architecture column: {column}"
        )

    ARCHITECTURE_SUMMARY_DF[column] = (
        ARCHITECTURE_SUMMARY_DF[column]
        .astype(int)
    )

# --------------------------------------------------------------------------------------------------
# Validate architecture dimensions
# --------------------------------------------------------------------------------------------------

for row in ARCHITECTURE_SUMMARY_DF.itertuples():

    expected_generative_dimension = (
        row.numerical_features
        +
        row.categorical_features
        +
        1
    )

    if row.generative_dimension != (
        expected_generative_dimension
    ):
        raise RuntimeError(
            f"Generative dimension mismatch: {row.dataset}"
        )

CRITIC_INPUT_DIMENSIONS = {
    row.dataset: int(
        row.transformed_dimension
    )
    for row in ARCHITECTURE_SUMMARY_DF.itertuples()
}

LATENT_DIMENSION = 128

print(
    f"✓ Architecture summary:\n"
    f"  {ARCHITECTURE_SUMMARY_PATH}"
)

display(
    ARCHITECTURE_SUMMARY_DF[
        [
            "dataset",
            "generative_dimension",
            "transformed_dimension",
            "numerical_features",
            "categorical_features",
        ]
    ]
)

print(
    f"✓ Latent dimension : {LATENT_DIMENSION}"
)

print(
    "✓ Critic dimensions:"
)

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id:18s}: "
        f"{CRITIC_INPUT_DIMENSIONS[dataset_id]}"
    )

print("SECTION 3 STATUS: PASS")


3. LOAD SPP-GAN ARCHITECTURE


NameError: name 'NB08_ROOT' is not defined

In [8]:
# ==================================================================================================
# 4. LOAD PRIVACY CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("4. LOAD PRIVACY CONFIGURATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Opacus
# --------------------------------------------------------------------------------------------------

if importlib.util.find_spec(
    "opacus"
) is None:

    print(
        "Opacus not installed. Installing..."
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "opacus",
        ]
    )

import opacus

from opacus import PrivacyEngine
from opacus.grad_sample import GradSampleModule
from opacus.validators import ModuleValidator
from opacus.accountants.utils import get_noise_multiplier

print(
    f"✓ Opacus version : {opacus.__version__}"
)

# --------------------------------------------------------------------------------------------------
# Methodology-aligned privacy parameters
# --------------------------------------------------------------------------------------------------

TARGET_EPSILON = 5.0

MAX_GRAD_NORM = 1.0

DP_BATCH_SIZE = 128

DP_EPOCHS = 300

ACCOUNTANT = "rdp"

SAMPLING_MECHANISM = "poisson"

CLIPPING_MECHANISM = "flat"

LOSS_REDUCTION = "mean"

GRAD_SAMPLE_MODE = "hooks"

# The generator is not independently privatized in this notebook.
GENERATOR_PRIVATE = False

# Statistical guidance remains outside the DP mechanism here.
STATISTICAL_GUIDANCE_PRIVATE = False

# Preprocessing remains outside this DP mechanism here.
PREPROCESSING_PRIVATE = False

# --------------------------------------------------------------------------------------------------
# Dataset-specific delta
# --------------------------------------------------------------------------------------------------

DELTA_BY_DATASET = {
    dataset_id: min(
        1e-5,
        1.0 / TRAIN_ROWS[dataset_id]
    )
    for dataset_id in DATASET_IDS
}

PRIVACY_CONFIGURATION = {
    "mechanism": "DP-SGD",
    "protected_component": "SPP-GAN discriminator",
    "target_epsilon": TARGET_EPSILON,
    "delta_rule": "min(1e-5, 1/N_train)",
    "max_grad_norm": MAX_GRAD_NORM,
    "batch_size": DP_BATCH_SIZE,
    "epochs": DP_EPOCHS,
    "accountant": ACCOUNTANT,
    "sampling": SAMPLING_MECHANISM,
    "clipping": CLIPPING_MECHANISM,
    "loss_reduction": LOSS_REDUCTION,
    "grad_sample_mode": GRAD_SAMPLE_MODE,
    "generator_private": GENERATOR_PRIVATE,
    "statistical_guidance_private": STATISTICAL_GUIDANCE_PRIVATE,
    "preprocessing_private": PREPROCESSING_PRIVATE,
    "end_to_end_privacy_claim": False,
}

print()
print("Privacy configuration:")

for key, value in PRIVACY_CONFIGURATION.items():
    print(
        f"  {key:40s}: {value}"
    )

print()
print("Dataset-specific δ:")

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id:18s}: "
        f"{DELTA_BY_DATASET[dataset_id]:.12g}"
    )

print("SECTION 4 STATUS: PASS")


4. LOAD PRIVACY CONFIGURATION
Opacus not installed. Installing...
✓ Opacus version : 1.6.0


NameError: name 'DATASET_IDS' is not defined

In [9]:
# ==================================================================================================
# 5. VALIDATE PRIVACY PARAMETERS
# ==================================================================================================

print("\n" + "=" * 100)
print("5. VALIDATE PRIVACY PARAMETERS")
print("=" * 100)

if TARGET_EPSILON <= 0:
    raise ValueError(
        "Target epsilon must be positive."
    )

if MAX_GRAD_NORM <= 0:
    raise ValueError(
        "Maximum gradient norm must be positive."
    )

if DP_BATCH_SIZE <= 0:
    raise ValueError(
        "DP batch size must be positive."
    )

if DP_EPOCHS <= 0:
    raise ValueError(
        "DP epochs must be positive."
    )

if ACCOUNTANT != "rdp":
    raise ValueError(
        "RDP accountant required."
    )

if SAMPLING_MECHANISM != "poisson":
    raise ValueError(
        "Poisson sampling required."
    )

if CLIPPING_MECHANISM != "flat":
    raise ValueError(
        "Flat clipping required."
    )

if LOSS_REDUCTION != "mean":
    raise ValueError(
        "Mean loss reduction required."
    )

PRIVACY_PARAMETER_ROWS = []

for dataset_id in DATASET_IDS:

    n_train = int(
        TRAIN_ROWS[dataset_id]
    )

    delta = float(
        DELTA_BY_DATASET[dataset_id]
    )

    sample_rate = (
        DP_BATCH_SIZE /
        n_train
    )

    if not (
        0 < sample_rate < 1
    ):
        raise ValueError(
            f"Invalid sample rate for {dataset_id}: "
            f"{sample_rate}"
        )

    noise_multiplier = get_noise_multiplier(
        target_epsilon=TARGET_EPSILON,
        target_delta=delta,
        sample_rate=sample_rate,
        epochs=DP_EPOCHS,
        accountant=ACCOUNTANT,
        epsilon_tolerance=0.001,
    )

    if (
        not np.isfinite(
            noise_multiplier
        )
        or noise_multiplier <= 0
    ):
        raise RuntimeError(
            f"Invalid noise multiplier for {dataset_id}."
        )

    PRIVACY_PARAMETER_ROWS.append({
        "dataset": dataset_id,
        "n_train": n_train,
        "target_epsilon": TARGET_EPSILON,
        "delta": delta,
        "batch_size": DP_BATCH_SIZE,
        "sample_rate": sample_rate,
        "epochs": DP_EPOCHS,
        "max_grad_norm": MAX_GRAD_NORM,
        "noise_multiplier": float(
            noise_multiplier
        ),
        "accountant": ACCOUNTANT,
        "sampling": SAMPLING_MECHANISM,
        "clipping": CLIPPING_MECHANISM,
        "status": "PASS",
    })

PRIVACY_PARAMETER_DF = pd.DataFrame(
    PRIVACY_PARAMETER_ROWS
)

if not (
    PRIVACY_PARAMETER_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Privacy parameter validation failed."
    )

print("✓ Privacy parameters validated.")

display(
    PRIVACY_PARAMETER_DF
)

print("SECTION 5 STATUS: PASS")


5. VALIDATE PRIVACY PARAMETERS


NameError: name 'DATASET_IDS' is not defined

In [10]:
# ==================================================================================================
# 6. DEFINE PER-EXAMPLE DISCRIMINATOR GRADIENTS
# ==================================================================================================

print("\n" + "=" * 100)
print("6. DEFINE PER-EXAMPLE DISCRIMINATOR GRADIENTS")
print("=" * 100)

class SPPGANCritic(
    nn.Module
):
    """
    Notebook 08-compatible SPP-GAN critic.

    Input:
        transformed tabular representation

    Architecture:
        Linear -> LeakyReLU
        Linear -> LeakyReLU
        Linear -> scalar critic score
    """

    def __init__(
        self,
        input_dim,
    ):
        super().__init__()

        self.input_dim = int(
            input_dim
        )

        self.network = nn.Sequential(
            nn.Linear(
                self.input_dim,
                256,
            ),
            nn.LeakyReLU(
                negative_slope=0.2
            ),
            nn.Linear(
                256,
                256,
            ),
            nn.LeakyReLU(
                negative_slope=0.2
            ),
            nn.Linear(
                256,
                1,
            ),
        )

    def forward(
        self,
        x,
    ):
        return self.network(x)


def validate_critic_for_dp(
    critic,
):
    """
    Validate that the critic is compatible with Opacus.
    """

    errors = ModuleValidator.validate(
        critic,
        strict=False,
    )

    if errors:
        raise RuntimeError(
            "Critic is not compatible with Opacus:\n"
            +
            "\n".join(
                str(error)
                for error in errors
            )
        )

    return True


def wrap_critic_for_per_sample_gradients(
    critic,
):
    """
    Wrap the SPP-GAN critic with Opacus
    per-sample gradient support.
    """

    validate_critic_for_dp(
        critic
    )

    return GradSampleModule(
        critic,
        batch_first=True,
        loss_reduction="mean",
        strict=True,
        force_functorch=False,
    )


def get_per_example_gradients(
    dp_critic,
):
    """
    Extract per-example gradients from the
    Opacus GradSampleModule.
    """

    gradients = {}

    for name, parameter in (
        dp_critic.named_parameters()
    ):

        grad_sample = getattr(
            parameter,
            "grad_sample",
            None,
        )

        if grad_sample is None:
            raise RuntimeError(
                f"Missing grad_sample for parameter: {name}"
            )

        if isinstance(
            grad_sample,
            list,
        ):

            if len(grad_sample) != 1:
                raise RuntimeError(
                    f"Unexpected accumulated grad_sample for {name}."
                )

            grad_sample = (
                grad_sample[0]
            )

        gradients[name] = grad_sample

    return gradients


print(
    "✓ SPP-GAN critic defined from Notebook 08 architecture."
)

print(
    "✓ Opacus per-example gradient interface defined."
)

print("SECTION 6 STATUS: PASS")


6. DEFINE PER-EXAMPLE DISCRIMINATOR GRADIENTS
✓ SPP-GAN critic defined from Notebook 08 architecture.
✓ Opacus per-example gradient interface defined.
SECTION 6 STATUS: PASS


In [11]:
# ==================================================================================================
# 7. DEFINE GRADIENT CLIPPING
# ==================================================================================================

print("\n" + "=" * 100)
print("7. DEFINE GRADIENT CLIPPING")
print("=" * 100)


def per_example_gradient_norms(
    gradients,
):
    """
    Compute the flat L2 norm across all discriminator
    parameters for each example.
    """

    if not gradients:
        raise ValueError(
            "No gradients supplied."
        )

    first_gradient = next(
        iter(
            gradients.values()
        )
    )

    batch_size = (
        first_gradient.shape[0]
    )

    squared_norms = torch.zeros(
        batch_size,
        device=first_gradient.device,
        dtype=first_gradient.dtype,
    )

    for gradient in gradients.values():

        flat_gradient = gradient.reshape(
            batch_size,
            -1,
        )

        squared_norms += (
            flat_gradient
            .pow(2)
            .sum(dim=1)
        )

    return torch.sqrt(
        squared_norms.clamp_min(
            0.0
        )
    )


def flat_clipping_factors(
    gradients,
    max_grad_norm,
):
    """
    Compute flat L2 clipping factors.
    """

    if max_grad_norm <= 0:
        raise ValueError(
            "max_grad_norm must be positive."
        )

    norms = per_example_gradient_norms(
        gradients
    )

    factors = (
        max_grad_norm /
        (norms + 1e-12)
    ).clamp(
        max=1.0
    )

    return factors


def clip_per_example_gradients(
    gradients,
    max_grad_norm,
):
    """
    Apply flat per-example L2 clipping.
    """

    factors = flat_clipping_factors(
        gradients,
        max_grad_norm,
    )

    clipped_gradients = {}

    for name, gradient in gradients.items():

        view_shape = (
            [factors.shape[0]]
            +
            [1] * (
                gradient.ndim - 1
            )
        )

        clipped_gradients[name] = (
            gradient
            *
            factors.reshape(
                view_shape
            )
        )

    return clipped_gradients


print(
    "✓ Flat L2 clipping defined."
)

print(
    f"✓ Maximum gradient norm C = {MAX_GRAD_NORM}"
)

print("SECTION 7 STATUS: PASS")


7. DEFINE GRADIENT CLIPPING
✓ Flat L2 clipping defined.
✓ Maximum gradient norm C = 1.0
SECTION 7 STATUS: PASS


In [12]:
# ==================================================================================================
# 7. DEFINE GRADIENT CLIPPING
# ==================================================================================================

print("\n" + "=" * 100)
print("7. DEFINE GRADIENT CLIPPING")
print("=" * 100)


def per_example_gradient_norms(
    gradients,
):
    """
    Compute the flat L2 norm across all discriminator
    parameters for each example.
    """

    if not gradients:
        raise ValueError(
            "No gradients supplied."
        )

    first_gradient = next(
        iter(
            gradients.values()
        )
    )

    batch_size = (
        first_gradient.shape[0]
    )

    squared_norms = torch.zeros(
        batch_size,
        device=first_gradient.device,
        dtype=first_gradient.dtype,
    )

    for gradient in gradients.values():

        flat_gradient = gradient.reshape(
            batch_size,
            -1,
        )

        squared_norms += (
            flat_gradient
            .pow(2)
            .sum(dim=1)
        )

    return torch.sqrt(
        squared_norms.clamp_min(
            0.0
        )
    )


def flat_clipping_factors(
    gradients,
    max_grad_norm,
):
    """
    Compute flat L2 clipping factors.
    """

    if max_grad_norm <= 0:
        raise ValueError(
            "max_grad_norm must be positive."
        )

    norms = per_example_gradient_norms(
        gradients
    )

    factors = (
        max_grad_norm /
        (norms + 1e-12)
    ).clamp(
        max=1.0
    )

    return factors


def clip_per_example_gradients(
    gradients,
    max_grad_norm,
):
    """
    Apply flat per-example L2 clipping.
    """

    factors = flat_clipping_factors(
        gradients,
        max_grad_norm,
    )

    clipped_gradients = {}

    for name, gradient in gradients.items():

        view_shape = (
            [factors.shape[0]]
            +
            [1] * (
                gradient.ndim - 1
            )
        )

        clipped_gradients[name] = (
            gradient
            *
            factors.reshape(
                view_shape
            )
        )

    return clipped_gradients


print(
    "✓ Flat L2 clipping defined."
)

print(
    f"✓ Maximum gradient norm C = {MAX_GRAD_NORM}"
)

print("SECTION 7 STATUS: PASS")


7. DEFINE GRADIENT CLIPPING
✓ Flat L2 clipping defined.
✓ Maximum gradient norm C = 1.0
SECTION 7 STATUS: PASS


In [13]:
# ==================================================================================================
# 9. DEFINE DP-SGD DISCRIMINATOR UPDATE
# ==================================================================================================

print("\n" + "=" * 100)
print("9. DEFINE DP-SGD DISCRIMINATOR UPDATE")
print("=" * 100)


def make_private_discriminator(
    critic,
    optimizer,
    data_loader,
    noise_multiplier,
    max_grad_norm,
):
    """
    Attach Opacus DP-SGD to the SPP-GAN discriminator.

    Privacy responsibilities:
        1. per-example gradients
        2. flat clipping
        3. Gaussian noise
        4. Poisson sampling
        5. privacy accountant state

    Formal accounting is finalized in Notebook 11.
    """

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    (
        private_critic,
        private_optimizer,
        private_loader,
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=optimizer,
        data_loader=data_loader,
        noise_multiplier=float(
            noise_multiplier
        ),
        max_grad_norm=float(
            max_grad_norm
        ),
        batch_first=True,
        loss_reduction="mean",
        poisson_sampling=True,
        clipping="flat",
        grad_sample_mode="hooks",
        wrap_model=True,
    )

    return (
        private_critic,
        private_optimizer,
        private_loader,
        privacy_engine,
    )


def dp_discriminator_loss(
    private_critic,
    real_batch,
    fake_batch,
):
    """
    WGAN-style per-example discriminator loss:

        l_i = D(fake_i) - D(real_i)

    Final batch loss:

        L_D = mean_i(l_i)

    fake_batch must be detached from the generator
    during the discriminator update.
    """

    if real_batch.shape != fake_batch.shape:
        raise ValueError(
            "Real and fake batches must have identical shapes."
        )

    real_scores = (
        private_critic(
            real_batch
        )
        .reshape(-1)
    )

    fake_scores = (
        private_critic(
            fake_batch.detach()
        )
        .reshape(-1)
    )

    per_example_loss = (
        fake_scores
        -
        real_scores
    )

    loss = (
        per_example_loss.mean()
    )

    return (
        loss,
        per_example_loss,
    )


def dp_discriminator_step(
    private_critic,
    private_optimizer,
    real_batch,
    fake_batch,
):
    """
    Execute one DP discriminator update.

    Opacus performs:
        per-example gradient calculation
        flat gradient clipping
        Gaussian noise addition
        optimizer update
    """

    private_optimizer.zero_grad(
        set_to_none=True
    )

    loss, per_example_loss = (
        dp_discriminator_loss(
            private_critic=private_critic,
            real_batch=real_batch,
            fake_batch=fake_batch,
        )
    )

    loss.backward()

    private_optimizer.step()

    return {
        "loss": loss.detach(),
        "per_example_loss": (
            per_example_loss.detach()
        ),
    }


print(
    "✓ DP-SGD discriminator update defined."
)

print(
    "✓ WGAN-style per-example critic objective defined."
)

print(
    "✓ Clipping/noise delegated to Opacus."
)

print("SECTION 9 STATUS: PASS")


9. DEFINE DP-SGD DISCRIMINATOR UPDATE
✓ DP-SGD discriminator update defined.
✓ WGAN-style per-example critic objective defined.
✓ Clipping/noise delegated to Opacus.
SECTION 9 STATUS: PASS


In [14]:
# ==================================================================================================
# 10. DEFINE SAMPLING MECHANISM
# ==================================================================================================

print("\n" + "=" * 100)
print("10. DEFINE SAMPLING MECHANISM")
print("=" * 100)


def poisson_sample_indices(
    n_samples,
    sample_rate,
    generator=None,
):
    """
    Diagnostic Poisson/Bernoulli inclusion sampler.

    Each record is independently included with
    probability q = sample_rate.

    Production training uses Opacus DPDataLoader.
    """

    if n_samples <= 0:
        raise ValueError(
            "n_samples must be positive."
        )

    if not (
        0 < sample_rate <= 1
    ):
        raise ValueError(
            "sample_rate must be in (0, 1]."
        )

    included = (
        torch.rand(
            n_samples,
            generator=generator,
            device="cpu",
        )
        < sample_rate
    )

    return torch.nonzero(
        included,
        as_tuple=False,
    ).flatten()


def validate_poisson_sampling(
    n_samples,
    sample_rate,
    repetitions=100,
):
    """
    RAM-safe diagnostic for Poisson batch-size behavior.
    """

    generator = torch.Generator(
        device="cpu"
    )

    generator.manual_seed(
        MASTER_SEED
    )

    batch_sizes = []

    for _ in range(
        repetitions
    ):

        indices = poisson_sample_indices(
            n_samples=n_samples,
            sample_rate=sample_rate,
            generator=generator,
        )

        batch_sizes.append(
            int(
                indices.numel()
            )
        )

    batch_sizes = np.asarray(
        batch_sizes,
        dtype=np.int64,
    )

    return {
        "mean_batch_size": float(
            batch_sizes.mean()
        ),
        "min_batch_size": int(
            batch_sizes.min()
        ),
        "max_batch_size": int(
            batch_sizes.max()
        ),
        "expected_batch_size": float(
            n_samples *
            sample_rate
        ),
        "repetitions": repetitions,
    }


print(
    "✓ Diagnostic Poisson sampler defined."
)

print(
    "✓ Production sampling mechanism: "
    "Opacus DPDataLoader"
)

print("SECTION 10 STATUS: PASS")


10. DEFINE SAMPLING MECHANISM
✓ Diagnostic Poisson sampler defined.
✓ Production sampling mechanism: Opacus DPDataLoader
SECTION 10 STATUS: PASS


In [15]:
# ==================================================================================================
# 11. TEST GRADIENT PRIVATIZATION
# ==================================================================================================

print("\n" + "=" * 100)
print("11. TEST GRADIENT PRIVATIZATION")
print("=" * 100)

GRADIENT_TEST_ROWS = []

TEST_BATCH_SIZE = 8

for row in ARCHITECTURE_SUMMARY_DF.itertuples():

    dataset_id = row.dataset

    transformed_dimension = int(
        row.transformed_dimension
    )

    torch.manual_seed(
        MASTER_SEED
    )

    critic = SPPGANCritic(
        transformed_dimension
    ).to(
        DEVICE
    )

    dp_critic = (
        wrap_critic_for_per_sample_gradients(
            critic
        )
    )

    dp_critic.train()

    real_batch = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
    )

    fake_batch = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
    )

    dp_critic.zero_grad(
        set_to_none=True
    )

    loss, per_example_loss = (
        dp_discriminator_loss(
            private_critic=dp_critic,
            real_batch=real_batch,
            fake_batch=fake_batch,
        )
    )

    if per_example_loss.shape[0] != (
        TEST_BATCH_SIZE
    ):
        raise RuntimeError(
            f"Invalid per-example loss size for {dataset_id}."
        )

    loss.backward()

    gradients = (
        get_per_example_gradients(
            dp_critic
        )
    )

    norms = (
        per_example_gradient_norms(
            gradients
        )
    )

    clipped_gradients = (
        clip_per_example_gradients(
            gradients,
            MAX_GRAD_NORM,
        )
    )

    clipped_norms = (
        per_example_gradient_norms(
            clipped_gradients
        )
    )

    shape_pass = all(
        gradient.shape[0]
        == TEST_BATCH_SIZE
        for gradient in gradients.values()
    )

    finite_pass = all(
        torch.isfinite(
            gradient
        ).all().item()
        for gradient in gradients.values()
    )

    clipping_pass = (
        torch.isfinite(
            clipped_norms
        ).all().item()
        and
        float(
            clipped_norms.max()
            .detach()
            .cpu()
        )
        <= (
            MAX_GRAD_NORM
            + 1e-5
        )
    )

    nonzero_pass = (
        float(
            norms.max()
            .detach()
            .cpu()
        )
        > 0
    )

    status = (
        "PASS"
        if all([
            shape_pass,
            finite_pass,
            clipping_pass,
            nonzero_pass,
        ])
        else "FAIL"
    )

    GRADIENT_TEST_ROWS.append({
        "dataset": dataset_id,
        "transformed_dimension": transformed_dimension,
        "batch_size": TEST_BATCH_SIZE,
        "gradient_parameter_count": len(
            gradients
        ),
        "max_raw_gradient_norm": float(
            norms.max()
            .detach()
            .cpu()
        ),
        "max_clipped_gradient_norm": float(
            clipped_norms.max()
            .detach()
            .cpu()
        ),
        "shape_validation": (
            "PASS"
            if shape_pass
            else "FAIL"
        ),
        "finite_validation": (
            "PASS"
            if finite_pass
            else "FAIL"
        ),
        "clipping_validation": (
            "PASS"
            if clipping_pass
            else "FAIL"
        ),
        "nonzero_validation": (
            "PASS"
            if nonzero_pass
            else "FAIL"
        ),
        "status": status,
    })

    del (
        dp_critic,
        critic,
        real_batch,
        fake_batch,
        gradients,
        clipped_gradients,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

GRADIENT_TEST_DF = pd.DataFrame(
    GRADIENT_TEST_ROWS
)

if not (
    GRADIENT_TEST_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Gradient privatization validation failed."
    )

GRADIENT_TEST_PATH = (
    DIRS["validation"] /
    "dp_gradient_privatization_tests.csv"
)

GRADIENT_TEST_DF.to_csv(
    GRADIENT_TEST_PATH,
    index=False,
)

display(
    GRADIENT_TEST_DF
)

print(
    f"✓ Gradient privatization tests saved:\n"
    f"  {GRADIENT_TEST_PATH}"
)

print(
    "SECTION 11 STATUS: PASS"
)


11. TEST GRADIENT PRIVATIZATION


NameError: name 'ARCHITECTURE_SUMMARY_DF' is not defined

In [16]:
# ==================================================================================================
# 12. TEST NOISE INJECTION
# ==================================================================================================

print("\n" + "=" * 100)
print("12. TEST NOISE INJECTION")
print("=" * 100)

NOISE_TEST_ROWS = []

NOISE_TEST_SIZE = 8192

noise_generator = torch.Generator(
    device=DEVICE.type
)

noise_generator.manual_seed(
    MASTER_SEED
)

for row in PRIVACY_PARAMETER_DF.itertuples():

    reference = torch.zeros(
        NOISE_TEST_SIZE,
        device=DEVICE,
        dtype=torch.float32,
    )

    noise = gaussian_noise(
        reference_tensor=reference,
        noise_multiplier=row.noise_multiplier,
        max_grad_norm=MAX_GRAD_NORM,
        generator=noise_generator,
    )

    expected_std = (
        row.noise_multiplier
        *
        MAX_GRAD_NORM
    )

    observed_std = float(
        noise.std(
            unbiased=True
        )
        .detach()
        .cpu()
    )

    finite_pass = bool(
        torch.isfinite(
            noise
        ).all().item()
    )

    nonzero_fraction = float(
        (
            noise != 0
        )
        .float()
        .mean()
        .detach()
        .cpu()
    )

    nonzero_pass = (
        nonzero_fraction > 0
    )

    relative_error = (
        abs(
            observed_std
            -
            expected_std
        )
        /
        expected_std
    )

    scale_pass = (
        relative_error < 0.15
    )

    status = (
        "PASS"
        if all([
            finite_pass,
            nonzero_pass,
            scale_pass,
        ])
        else "FAIL"
    )

    NOISE_TEST_ROWS.append({
        "dataset": row.dataset,
        "noise_multiplier": row.noise_multiplier,
        "expected_std": expected_std,
        "observed_std": observed_std,
        "relative_std_error": relative_error,
        "finite": finite_pass,
        "nonzero": nonzero_pass,
        "scale_validation": scale_pass,
        "status": status,
    })

    del (
        reference,
        noise,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

NOISE_TEST_DF = pd.DataFrame(
    NOISE_TEST_ROWS
)

if not (
    NOISE_TEST_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Gaussian noise validation failed."
    )

NOISE_TEST_PATH = (
    DIRS["validation"] /
    "dp_gaussian_noise_tests.csv"
)

NOISE_TEST_DF.to_csv(
    NOISE_TEST_PATH,
    index=False,
)

display(
    NOISE_TEST_DF
)

print(
    f"✓ Noise tests saved:\n"
    f"  {NOISE_TEST_PATH}"
)

print("SECTION 12 STATUS: PASS")


12. TEST NOISE INJECTION


NameError: name 'DEVICE' is not defined

In [17]:
# ==================================================================================================
# 13. VALIDATE PRIVACY MECHANISM
# ==================================================================================================

print("\n" + "=" * 100)
print("13. VALIDATE PRIVACY MECHANISM")
print("=" * 100)

PRIVACY_MECHANISM_ROWS = []

for row in ARCHITECTURE_SUMMARY_DF.itertuples():

    dataset_id = row.dataset

    transformed_dimension = int(
        row.transformed_dimension
    )

    # ----------------------------------------------------------------------------------------------
    # Critic
    # ----------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        transformed_dimension
    ).to(
        DEVICE
    )

    critic.train()

    validation_errors = (
        ModuleValidator.validate(
            critic,
            strict=False,
        )
    )

    critic_compatible = (
        len(validation_errors) == 0
    )

    # ----------------------------------------------------------------------------------------------
    # Dummy training data
    # ----------------------------------------------------------------------------------------------

    dummy_rows = 32

    dummy_x = torch.randn(
        dummy_rows,
        transformed_dimension,
    )

    dummy_dataset = (
        torch.utils.data.TensorDataset(
            dummy_x
        )
    )

    dummy_loader = (
        torch.utils.data.DataLoader(
            dummy_dataset,
            batch_size=8,
            shuffle=False,
            num_workers=0,
            pin_memory=False,
        )
    )

    optimizer = torch.optim.Adam(
        critic.parameters(),
        lr=2e-4,
    )

    # ----------------------------------------------------------------------------------------------
    # PrivacyEngine compatibility
    # ----------------------------------------------------------------------------------------------

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    try:

        engine_compatible = bool(
            privacy_engine.is_compatible(
                module=critic,
                optimizer=optimizer,
                data_loader=dummy_loader,
            )
        )

    except Exception:

        engine_compatible = False

    # ----------------------------------------------------------------------------------------------
    # Poisson diagnostic
    # ----------------------------------------------------------------------------------------------

    sample_rate = (
        DP_BATCH_SIZE
        /
        TRAIN_ROWS[dataset_id]
    )

    poisson_test = (
        validate_poisson_sampling(
            n_samples=TRAIN_ROWS[dataset_id],
            sample_rate=sample_rate,
            repetitions=50,
        )
    )

    poisson_pass = (
        poisson_test[
            "mean_batch_size"
        ] > 0
        and
        poisson_test[
            "mean_batch_size"
        ] < TRAIN_ROWS[dataset_id]
        and
        poisson_test[
            "max_batch_size"
        ] > 0
    )

    # ----------------------------------------------------------------------------------------------
    # Actual Opacus wrapping test
    # ----------------------------------------------------------------------------------------------

    try:

        (
            private_critic,
            private_optimizer,
            private_loader,
        ) = privacy_engine.make_private(
            module=critic,
            optimizer=optimizer,
            data_loader=dummy_loader,
            noise_multiplier=float(
                PRIVACY_PARAMETER_DF.loc[
                    PRIVACY_PARAMETER_DF[
                        "dataset"
                    ]
                    == dataset_id,
                    "noise_multiplier",
                ].iloc[0]
            ),
            max_grad_norm=MAX_GRAD_NORM,
            batch_first=True,
            loss_reduction="mean",
            poisson_sampling=True,
            clipping="flat",
            grad_sample_mode="hooks",
            wrap_model=True,
        )

        wrapping_pass = (
            private_critic is not None
            and
            private_optimizer is not None
            and
            private_loader is not None
        )

    except Exception as exc:

        wrapping_pass = False

        print(
            f"⚠ DP wrapping failed for {dataset_id}: "
            f"{exc}"
        )

    status = (
        "PASS"
        if all([
            critic_compatible,
            engine_compatible,
            poisson_pass,
            wrapping_pass,
        ])
        else "FAIL"
    )

    PRIVACY_MECHANISM_ROWS.append({
        "dataset": dataset_id,
        "transformed_dimension": transformed_dimension,
        "critic_compatible": critic_compatible,
        "privacy_engine_compatible": engine_compatible,
        "poisson_sampling": poisson_pass,
        "dp_wrapping": wrapping_pass,
        "status": status,
    })

    del (
        critic,
        optimizer,
        dummy_loader,
        dummy_dataset,
        dummy_x,
        privacy_engine,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

PRIVACY_MECHANISM_DF = pd.DataFrame(
    PRIVACY_MECHANISM_ROWS
)

if not (
    PRIVACY_MECHANISM_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Privacy mechanism validation failed."
    )

PRIVACY_MECHANISM_PATH = (
    DIRS["validation"] /
    "dp_privacy_mechanism_validation.csv"
)

PRIVACY_MECHANISM_DF.to_csv(
    PRIVACY_MECHANISM_PATH,
    index=False,
)

display(
    PRIVACY_MECHANISM_DF
)

print(
    f"✓ Privacy mechanism validation saved:\n"
    f"  {PRIVACY_MECHANISM_PATH}"
)

print("SECTION 13 STATUS: PASS")


13. VALIDATE PRIVACY MECHANISM


NameError: name 'ARCHITECTURE_SUMMARY_DF' is not defined

In [18]:
# ==================================================================================================
# 14. RECORD PRIVACY METADATA
# ==================================================================================================

print("\n" + "=" * 100)
print("14. RECORD PRIVACY METADATA")
print("=" * 100)

PRIVACY_METADATA_ROWS = []

for row in PRIVACY_PARAMETER_DF.itertuples():

    PRIVACY_METADATA_ROWS.append({

        "dataset":
            row.dataset,

        "n_train":
            row.n_train,

        "target_epsilon":
            row.target_epsilon,

        "delta":
            row.delta,

        "batch_size":
            row.batch_size,

        "sample_rate":
            row.sample_rate,

        "epochs":
            row.epochs,

        "max_grad_norm":
            row.max_grad_norm,

        "noise_multiplier":
            row.noise_multiplier,

        "accountant":
            row.accountant,

        "sampling":
            row.sampling,

        "clipping":
            row.clipping,

        "loss_reduction":
            LOSS_REDUCTION,

        "protected_component":
            "SPP-GAN discriminator",

        "per_example_gradients":
            True,

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "achieved_epsilon":
            None,

        "achieved_epsilon_status":
            "DEFERRED_TO_NOTEBOOK_11",

        "training_status":
            "DEFERRED_TO_NOTEBOOK_12",

        "synthetic_generation_status":
            "DEFERRED_TO_NOTEBOOK_13",

        "end_to_end_privacy_claim":
            False,

        "status":
            "PASS",
    })

PRIVACY_METADATA_DF = pd.DataFrame(
    PRIVACY_METADATA_ROWS
)

PRIVACY_METADATA_PATH = (
    DIRS["metadata"] /
    "sppgan_privacy_metadata.csv"
)

PRIVACY_METADATA_DF.to_csv(
    PRIVACY_METADATA_PATH,
    index=False,
)

display(
    PRIVACY_METADATA_DF
)

print(
    f"✓ Privacy metadata saved:\n"
    f"  {PRIVACY_METADATA_PATH}"
)

print("SECTION 14 STATUS: PASS")


14. RECORD PRIVACY METADATA


NameError: name 'PRIVACY_PARAMETER_DF' is not defined

In [19]:
# ==================================================================================================
# 15. SAVE DP MODULE
# ==================================================================================================

print("\n" + "=" * 100)
print("15. SAVE DP MODULE")
print("=" * 100)

DP_MODULE_PATH = (
    DIRS["models"] /
    "sppgan_differential_privacy.py"
)

DP_MODULE_SOURCE = r'''
# ==================================================================================================
# SPP-GAN DIFFERENTIAL PRIVACY MODULE
# ==================================================================================================

import torch
import torch.nn as nn

from opacus import PrivacyEngine
from opacus.grad_sample import GradSampleModule


class SPPGANCritic(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.input_dim = int(
            input_dim
        )

        self.network = nn.Sequential(
            nn.Linear(
                self.input_dim,
                256,
            ),
            nn.LeakyReLU(
                0.2
            ),
            nn.Linear(
                256,
                256,
            ),
            nn.LeakyReLU(
                0.2
            ),
            nn.Linear(
                256,
                1,
            ),
        )

    def forward(self, x):

        return self.network(x)


def wrap_critic_for_per_sample_gradients(
    critic,
):

    return GradSampleModule(
        critic,
        batch_first=True,
        loss_reduction="mean",
        strict=True,
        force_functorch=False,
    )


def dp_discriminator_loss(
    private_critic,
    real_batch,
    fake_batch,
):

    if real_batch.shape != fake_batch.shape:

        raise ValueError(
            "Real and fake batches must have identical shapes."
        )

    real_scores = (
        private_critic(
            real_batch
        )
        .reshape(-1)
    )

    fake_scores = (
        private_critic(
            fake_batch.detach()
        )
        .reshape(-1)
    )

    per_example_loss = (
        fake_scores
        -
        real_scores
    )

    return (
        per_example_loss.mean(),
        per_example_loss,
    )


def make_private_discriminator(
    critic,
    optimizer,
    data_loader,
    noise_multiplier,
    max_grad_norm,
):

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    (
        private_critic,
        private_optimizer,
        private_loader,
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=optimizer,
        data_loader=data_loader,
        noise_multiplier=float(
            noise_multiplier
        ),
        max_grad_norm=float(
            max_grad_norm
        ),
        batch_first=True,
        loss_reduction="mean",
        poisson_sampling=True,
        clipping="flat",
        grad_sample_mode="hooks",
        wrap_model=True,
    )

    return (
        private_critic,
        private_optimizer,
        private_loader,
        privacy_engine,
    )


def dp_discriminator_step(
    private_critic,
    private_optimizer,
    real_batch,
    fake_batch,
):

    private_optimizer.zero_grad(
        set_to_none=True
    )

    loss, per_example_loss = (
        dp_discriminator_loss(
            private_critic,
            real_batch,
            fake_batch,
        )
    )

    loss.backward()

    private_optimizer.step()

    return {
        "loss": loss.detach(),
        "per_example_loss":
            per_example_loss.detach(),
    }
'''

with open(
    DP_MODULE_PATH,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        DP_MODULE_SOURCE
    )

# --------------------------------------------------------------------------------------------------
# Syntax validation
# --------------------------------------------------------------------------------------------------

with open(
    DP_MODULE_PATH,
    "r",
    encoding="utf-8",
) as f:

    source = f.read()

compile(
    source,
    str(DP_MODULE_PATH),
    "exec",
)

print(
    f"✓ DP module saved:\n"
    f"  {DP_MODULE_PATH}"
)

print(
    "✓ Python syntax validation: PASS"
)

print("SECTION 15 STATUS: PASS")


15. SAVE DP MODULE


NameError: name 'DIRS' is not defined

In [20]:
# ==================================================================================================
# 16. SAVE PRIVACY CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("16. SAVE PRIVACY CONFIGURATION")
print("=" * 100)

PERSISTED_PRIVACY_CONFIGURATION = {

    "configuration_version":
        "1.0",

    "notebook":
        NOTEBOOK_ID,

    "name":
        NOTEBOOK_NAME,

    "framework":
        FRAMEWORK_NAME,

    "privacy_definition": {

        "mechanism":
            "DP-SGD",

        "protected_component":
            "SPP-GAN discriminator",

        "gradient_representation":
            "per-example discriminator gradients",

        "clipping":
            "flat L2",

        "noise":
            "Gaussian",

        "sampling":
            "Poisson",

        "accountant":
            "RDP",

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "end_to_end_privacy_claim":
            False,
    },

    "parameters": {

        "target_epsilon":
            TARGET_EPSILON,

        "delta_rule":
            "min(1e-5, 1/N_train)",

        "max_grad_norm":
            MAX_GRAD_NORM,

        "batch_size":
            DP_BATCH_SIZE,

        "epochs":
            DP_EPOCHS,

        "accountant":
            ACCOUNTANT,

        "sampling":
            SAMPLING_MECHANISM,

        "clipping":
            CLIPPING_MECHANISM,

        "loss_reduction":
            LOSS_REDUCTION,

        "grad_sample_mode":
            GRAD_SAMPLE_MODE,
    },

    "dataset_parameters":
        PRIVACY_PARAMETER_DF.to_dict(
            orient="records"
        ),

    "source_dependencies": {

        "notebook_08_architecture":
            str(
                NB08_ROOT
            ),

        "notebook_09_statistical_guidance":
            str(
                NB09_ROOT
            ),
    },

    "downstream": {

        "notebook_11":
            "Formal RDP privacy accounting",

        "notebook_12":
            "SPP-GAN DP training",

        "notebook_13":
            "Synthetic data generation",
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

PRIVACY_CONFIGURATION_PATH = (
    DIRS["configuration"] /
    "sppgan_privacy_configuration.json"
)

with open(
    PRIVACY_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        PERSISTED_PRIVACY_CONFIGURATION,
        f,
        indent=2,
        ensure_ascii=False,
    )

# --------------------------------------------------------------------------------------------------
# Reload
# --------------------------------------------------------------------------------------------------

with open(
    PRIVACY_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as f:

    RELOADED_PRIVACY_CONFIGURATION = json.load(
        f
    )

REQUIRED_PRIVACY_CONFIGURATION_KEYS = {
    "configuration_version",
    "notebook",
    "name",
    "framework",
    "privacy_definition",
    "parameters",
    "dataset_parameters",
    "source_dependencies",
    "downstream",
    "created_utc",
}

missing_keys = (
    REQUIRED_PRIVACY_CONFIGURATION_KEYS
    -
    set(
        RELOADED_PRIVACY_CONFIGURATION.keys()
    )
)

if missing_keys:
    raise RuntimeError(
        "Persisted privacy configuration is missing:\n"
        f"{sorted(missing_keys)}"
    )

print(
    f"✓ Privacy configuration saved:\n"
    f"  {PRIVACY_CONFIGURATION_PATH}"
)

print(
    "✓ Configuration reload validation: PASS"
)

print("SECTION 16 STATUS: PASS")


16. SAVE PRIVACY CONFIGURATION


NameError: name 'PRIVACY_PARAMETER_DF' is not defined

In [21]:
# ==================================================================================================
# 17. SAVE PRIVACY MANIFEST
# ==================================================================================================

print("\n" + "=" * 100)
print("17. SAVE PRIVACY MANIFEST")
print("=" * 100)


def sha256_file(
    path,
):
    """
    RAM-safe SHA-256 calculation.
    """

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


PRIVACY_ARTIFACTS = [
    DP_MODULE_PATH,
    PRIVACY_CONFIGURATION_PATH,
    PRIVACY_METADATA_PATH,
    GRADIENT_TEST_PATH,
    NOISE_TEST_PATH,
    PRIVACY_MECHANISM_PATH,
]

PRIVACY_ARTIFACT_REGISTRY = []

for artifact_path in PRIVACY_ARTIFACTS:

    artifact_path = Path(
        artifact_path
    )

    if not artifact_path.exists():

        raise FileNotFoundError(
            f"Required artifact missing:\n"
            f"{artifact_path}"
        )

    PRIVACY_ARTIFACT_REGISTRY.append({

        "artifact":
            artifact_path.name,

        "absolute_path":
            str(
                artifact_path
            ),

        "exists":
            True,

        "size_bytes":
            artifact_path.stat().st_size,

        "sha256":
            sha256_file(
                artifact_path
            ),

    })

PRIVACY_ARTIFACT_REGISTRY_DF = pd.DataFrame(
    PRIVACY_ARTIFACT_REGISTRY
)

PRIVACY_ARTIFACT_REGISTRY_PATH = (
    DIRS["metadata"] /
    "sppgan_privacy_artifact_registry.csv"
)

PRIVACY_ARTIFACT_REGISTRY_DF.to_csv(
    PRIVACY_ARTIFACT_REGISTRY_PATH,
    index=False,
)

display(
    PRIVACY_ARTIFACT_REGISTRY_DF
)

print(
    f"✓ Privacy artifact registry saved:\n"
    f"  {PRIVACY_ARTIFACT_REGISTRY_PATH}"
)

print(
    f"✓ Registered artifacts : "
    f"{len(PRIVACY_ARTIFACT_REGISTRY_DF)}"
)

print("SECTION 17 STATUS: PASS")


17. SAVE PRIVACY MANIFEST


NameError: name 'DP_MODULE_PATH' is not defined

In [22]:
# ==================================================================================================
# 18. FINAL VERIFICATION
# ==================================================================================================

print("\n" + "=" * 100)
print("18. FINAL VERIFICATION")
print("=" * 100)

FINAL_CHECKS = []

def add_final_check(
    name,
    condition,
):
    FINAL_CHECKS.append({
        "check": name,
        "status": (
            "PASS"
            if bool(condition)
            else "FAIL"
        ),
    })


# --------------------------------------------------------------------------------------------------
# Root / dependencies
# --------------------------------------------------------------------------------------------------

add_final_check(
    "Project root exists",
    PROJECT_ROOT.exists(),
)

add_final_check(
    "Notebook 08 architecture loaded",
    len(
        ARCHITECTURE_SUMMARY_DF
    ) == len(DATASET_IDS),
)

add_final_check(
    "Notebook 09 configuration loaded",
    NB09_CONFIGURATION_PATH.exists(),
)

# --------------------------------------------------------------------------------------------------
# Privacy parameters
# --------------------------------------------------------------------------------------------------

add_final_check(
    "Target epsilon positive",
    TARGET_EPSILON > 0,
)

add_final_check(
    "Maximum gradient norm positive",
    MAX_GRAD_NORM > 0,
)

add_final_check(
    "Poisson sampling selected",
    SAMPLING_MECHANISM == "poisson",
)

add_final_check(
    "Flat clipping selected",
    CLIPPING_MECHANISM == "flat",
)

add_final_check(
    "RDP accountant selected",
    ACCOUNTANT == "rdp",
)

# --------------------------------------------------------------------------------------------------
# Validation results
# --------------------------------------------------------------------------------------------------

add_final_check(
    "Gradient privatization tests pass",
    GRADIENT_TEST_DF[
        "status"
    ].eq("PASS").all(),
)

add_final_check(
    "Gaussian noise tests pass",
    NOISE_TEST_DF[
        "status"
    ].eq("PASS").all(),
)

add_final_check(
    "Privacy mechanism tests pass",
    PRIVACY_MECHANISM_DF[
        "status"
    ].eq("PASS").all(),
)

# --------------------------------------------------------------------------------------------------
# Artifacts
# --------------------------------------------------------------------------------------------------

add_final_check(
    "DP module exists",
    DP_MODULE_PATH.exists(),
)

add_final_check(
    "Privacy configuration exists",
    PRIVACY_CONFIGURATION_PATH.exists(),
)

add_final_check(
    "Privacy metadata exists",
    PRIVACY_METADATA_PATH.exists(),
)

add_final_check(
    "Artifact registry exists",
    PRIVACY_ARTIFACT_REGISTRY_PATH.exists(),
)

FINAL_CHECKS_DF = pd.DataFrame(
    FINAL_CHECKS
)

if not (
    FINAL_CHECKS_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):

    display(
        FINAL_CHECKS_DF
    )

    raise RuntimeError(
        "Notebook 10 final verification failed."
    )

FINAL_VERIFICATION_PATH = (
    DIRS["validation"] /
    "sppgan_privacy_final_verification.csv"
)

FINAL_CHECKS_DF.to_csv(
    FINAL_VERIFICATION_PATH,
    index=False,
)

display(
    FINAL_CHECKS_DF
)

print(
    f"✓ Final verification saved:\n"
    f"  {FINAL_VERIFICATION_PATH}"
)

print(
    "✓ All final checks: PASS"
)

print("SECTION 18 STATUS: PASS")


18. FINAL VERIFICATION


NameError: name 'ARCHITECTURE_SUMMARY_DF' is not defined

In [23]:
# ==================================================================================================
# 19. COMPLETION SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("19. COMPLETION SUMMARY")
print("=" * 100)

COMPLETION_MANIFEST = {

    "notebook":
        NOTEBOOK_ID,

    "name":
        NOTEBOOK_NAME,

    "framework":
        FRAMEWORK_NAME,

    "status":
        "PASS",

    "project_root":
        str(
            PROJECT_ROOT
        ),

    "datasets_registered":
        len(
            DATASET_IDS
        ),

    "datasets":
        DATASET_IDS,

    "privacy_mechanism": {

        "mechanism":
            "DP-SGD",

        "protected_component":
            "SPP-GAN discriminator",

        "per_example_gradients":
            "PASS",

        "flat_gradient_clipping":
            "PASS",

        "gaussian_noise":
            "PASS",

        "poisson_sampling":
            "PASS",

        "accountant":
            "RDP",
    },

    "parameters": {

        "target_epsilon":
            TARGET_EPSILON,

        "delta_rule":
            "min(1e-5, 1/N_train)",

        "max_grad_norm":
            MAX_GRAD_NORM,

        "batch_size":
            DP_BATCH_SIZE,

        "epochs":
            DP_EPOCHS,
    },

    "validation": {

        "architecture":
            "PASS",

        "privacy_parameters":
            "PASS",

        "gradient_privatization":
            "PASS",

        "noise_injection":
            "PASS",

        "privacy_mechanism":
            "PASS",

        "final_verification":
            "PASS",
    },

    "training_performed":
        False,

    "privacy_accounting_performed":
        False,

    "achieved_epsilon_reported":
        False,

    "synthetic_generation_performed":
        False,

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "end_to_end_privacy_claim":
        False,

    "artifacts": {

        "dp_module":
            str(
                DP_MODULE_PATH
            ),

        "privacy_configuration":
            str(
                PRIVACY_CONFIGURATION_PATH
            ),

        "privacy_metadata":
            str(
                PRIVACY_METADATA_PATH
            ),

        "gradient_tests":
            str(
                GRADIENT_TEST_PATH
            ),

        "noise_tests":
            str(
                NOISE_TEST_PATH
            ),

        "mechanism_validation":
            str(
                PRIVACY_MECHANISM_PATH
            ),

        "artifact_registry":
            str(
                PRIVACY_ARTIFACT_REGISTRY_PATH
            ),

        "final_verification":
            str(
                FINAL_VERIFICATION_PATH
            ),
    },

    "source_dependencies": {

        "notebook_08":
            str(
                NB08_ROOT
            ),

        "notebook_09":
            str(
                NB09_ROOT
            ),
    },

    "downstream": {

        "notebook_11":
            "SPP-GAN Privacy Accounting",

        "notebook_12":
            "SPP-GAN DP Training",

        "notebook_13":
            "SPP-GAN Synthetic Generation",
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

COMPLETION_PATH = (
    DIRS["validation"] /
    "sppgan_notebook_10_completion.json"
)

with open(
    COMPLETION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        COMPLETION_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
    )

# --------------------------------------------------------------------------------------------------
# Final display
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 10 — FINAL STATUS")
print("=" * 100)

print(
    f"Framework                       : "
    f"{FRAMEWORK_NAME}"
)

print(
    f"Datasets                        : "
    f"{len(DATASET_IDS)}"
)

print(
    f"Privacy mechanism               : DP-SGD"
)

print(
    f"Protected component             : "
    f"SPP-GAN discriminator"
)

print(
    f"Per-example gradients           : PASS"
)

print(
    f"Flat gradient clipping          : PASS"
)

print(
    f"Gaussian noise                  : PASS"
)

print(
    f"Poisson sampling                : PASS"
)

print(
    f"RDP accountant                  : CONFIGURED"
)

print(
    f"Target epsilon                  : "
    f"{TARGET_EPSILON}"
)

print(
    f"Maximum gradient norm           : "
    f"{MAX_GRAD_NORM}"
)

print(
    f"DP batch size                   : "
    f"{DP_BATCH_SIZE}"
)

print(
    f"DP epochs                       : "
    f"{DP_EPOCHS}"
)

print(
    f"Training                        : "
    f"NOT PERFORMED"
)

print(
    f"Privacy accounting              : "
    f"DEFERRED TO NOTEBOOK 11"
)

print(
    f"Achieved epsilon                : "
    f"NOT CLAIMED"
)

print(
    f"Synthetic generation            : "
    f"NOT PERFORMED"
)

print(
    f"End-to-end privacy claim        : "
    f"NOT ESTABLISHED"
)

print(
    f"Overall status                  : "
    f"PASS"
)

print()
print("Artifacts:")

print(
    f"  DP module       : "
    f"{DP_MODULE_PATH}"
)

print(
    f"  Configuration   : "
    f"{PRIVACY_CONFIGURATION_PATH}"
)

print(
    f"  Metadata        : "
    f"{PRIVACY_METADATA_PATH}"
)

print(
    f"  Manifest        : "
    f"{PRIVACY_ARTIFACT_REGISTRY_PATH}"
)

print(
    f"  Verification    : "
    f"{FINAL_VERIFICATION_PATH}"
)

print(
    f"  Completion      : "
    f"{COMPLETION_PATH}"
)

print()
print("Next:")

print(
    "  Notebook 11 — SPP-GAN Privacy Accounting"
)

print("=" * 100)

print(
    "\n✓ NOTEBOOK 10 COMPLETED SUCCESSFULLY."
)


19. COMPLETION SUMMARY


NameError: name 'DATASET_IDS' is not defined